In [5]:
import numpy as np
import pandas as pd
from pathlib import Path
import json
import time

pd.set_option("display.max_rows", 999)

In [6]:

def process_data(multi_omics_data: dict[str, Path], cell_key: str, dataname: str, prefix: Path, fast_edge=True,
                 read_csv_args: dict = {}):

    def get_omics_id_from_gene(gene_name: str):
        for id, omics_name in enumerate(multi_omics_data.keys()):
            if gene_name.endswith(omics_name):
                return id
        raise ValueError(gene_name + ' has no omics!')

    num_omics = len(multi_omics_data)
    print('Begin to load multi omics data', multi_omics_data)
    if not read_csv_args:
        read_csv_args = dict(index_col=cell_key)

    begin = time.time()
    multi_omics_df = {
        name: pd.read_csv(path, **read_csv_args).sort_index() for name, path in multi_omics_data.items()
    }
    print('Load data done, time:', time.time() - begin)

    # Rename feature columns.
    multi_omics_df = {
        name: df.rename({col: str(col).strip() + "_" + name for col in df.columns}, axis=1)
          for name, df in multi_omics_df.items()
    }
    print('Rename columns')
    for name, df in multi_omics_df.items():
        print('Name', name, 'Columns', df.columns)

    print('Union features from multi-omics')
    # Join the dataframes by cell key.
    df = pd.concat(list(multi_omics_df.values()), axis=1)

    # Remove all-NaNs columns
    df.dropna(axis=1, how='all', inplace=True)
    num_cells = len(df)
    num_genes = len(df.columns)
    num_nodes = num_cells + num_genes + num_omics

    print(f'{num_omics=}, {num_cells=}, {num_genes=}, {num_nodes=}')

    def add_node_type(nodes, node_type):
        return [(node, node_type) for node in nodes]

    omics_names = ['Omics_' + name for name in multi_omics_data.keys()]
    cell_names = df.index.values.tolist()
    gene_names = list(df.columns)
    node_names = add_node_type(omics_names, 0) + add_node_type(gene_names, 1) + \
        add_node_type(cell_names, 2)
    df_nodes = pd.DataFrame(node_names, columns=['Name', 'Type'])

    # Node offsets
    omics_offset = 0
    gene_offset = num_omics
    cell_offset = num_omics + num_genes

    def construct_hyper_edge():
        hyper_edges = []
        for cell_id in range(num_cells):
            for gene_id in range(num_genes):
                weight = df.iloc[cell_id, gene_id]
                if np.isnan(weight):
                    continue

                omics_id = get_omics_id_from_gene(gene_names[gene_id])
                omics_node = omics_id + omics_offset
                gene_node = gene_id + gene_offset
                cell_node = cell_id + cell_offset
                edge = dict(Omics=omics_node, Gene=gene_node,
                            Cell=cell_node, Weight=weight)
                hyper_edges.append(edge)

        df_edges =  pd.DataFrame.from_records(hyper_edges).astype(int)
        return df_edges

    def construct_hyper_edge_fast():

        # 0. 预计算 gene_id → omics_id
        gene2omics = pd.Series(
            [get_omics_id_from_gene(g) for g in df.columns],
            index=pd.RangeIndex(len(df.columns))   # 0,1,2,...
        )

        # 1. 把列名换成整数，再 stack
        df_int_cols = df.set_axis(range(df.shape[1]), axis=1)   # 关键一步
        w = df_int_cols.to_numpy()
        cell_id, gene_id = np.nonzero(~np.isnan(w))   # 非 NaN 的坐标
        weight = w[cell_id, gene_id]
        long = pd.DataFrame({'cell_id': cell_id,
                     'gene_id': gene_id,
                     'Weight': weight})

        # ---- 2. 三列 node id 向量化 ----
        long['Omics']  = gene2omics[long['gene_id']].values + omics_offset
        long['Gene']   = long['gene_id'].values + gene_offset
        long['Cell']   = long['cell_id'].values + cell_offset

        # ---- 3. 只要这四列 ----
        hyper_edges = long[['Omics', 'Gene', 'Cell', 'Weight']]
        return hyper_edges

    begin = time.time()
    print('Constructing hyper edges...')
    df_edges = construct_hyper_edge_fast() if fast_edge else construct_hyper_edge()
    print('Construct edges done, time:', time.time() - begin)

    savedir = Path(prefix) / dataname
    savedir.mkdir(parents=True, exist_ok=True)
    node_file = savedir / 'nodes.csv'
    df_nodes.to_csv(node_file, index=True)
    print('Save node file', node_file)

    edge_file = savedir / 'edges.csv'
    df_edges.to_csv(edge_file, index=False)
    print('Save edge file', edge_file)

    metadata = dict(
        num_cells=num_cells, num_genes=num_genes, num_omics=num_omics,
        num_nodes=num_nodes, num_edges=len(df_edges),
        omics_offset=omics_offset, gene_offset=gene_offset, cell_offset=cell_offset,
        raw_data=multi_omics_data,
        cell_key=cell_key,
    )

    metadata_file = savedir / 'metadata.json'
    json.dump(metadata, metadata_file.open('w'), indent=4)
    print('Save metadata', metadata_file)

In [7]:
process_data({
        'expr': './data/sc_GEM/expression_data.csv',
        'methy': './data/sc_GEM/methylation_data.csv',
    }, cell_key='Cell ID', dataname='sc_GEM', prefix=Path('./hypergraph'))

Begin to load multi omics data {'expr': './data/sc_GEM/expression_data.csv', 'methy': './data/sc_GEM/methylation_data.csv'}
Load data done, time: 0.02552485466003418
Rename columns
Name expr Columns Index(['ADAM33_expr', 'AFP_expr', 'AK123759_expr', 'ALDH3A1_expr', 'BMP7_expr',
       'CDH1_expr', 'CDH22_expr', 'CDX2_expr', 'CER1_expr', 'CHL1_expr',
       'COL20A1_expr', 'COL23A1_expr', 'CYBRD1_expr', 'DAZL_expr',
       'DNMT3B_expr', 'DNMT3L_expr', 'DPPA3_expr', 'EPHB3_expr', 'FGF4_expr',
       'FOSL1_expr', 'FOXD2_expr', 'HAND1_expr', 'HLX-AS1_expr', 'IGF2_expr',
       'JARID2_expr', 'KCNQ2_expr', 'LEFTY_expr', 'LTBR_expr', 'LUM_expr',
       'LY86-AS1_expr', 'MMP9_expr', 'MYC_expr', 'NANOG_expr', 'NESTIN_expr',
       'NFATC1_expr', 'NFIX_expr', 'NKX2-5_expr', 'NXRA8_expr', 'OCT4_expr',
       'OTX2_expr', 'PIWIL1_expr', 'PLEKHH3_expr', 'PRDM14_expr', 'SALL4_expr',
       'SOX2_expr', 'STON2P1_expr', 'SULT1A1_expr', 'TBX3_expr', 'TFAP2A_expr',
       'TFCP2L1_expr', 'TGFBR2_expr

In [8]:
process_data({
        'expr': './data/PEA_STA/expression_data.csv',
        'protein': './data/PEA_STA/protein_data.csv',
    }, cell_key='Cells ', dataname='PEA_STA', prefix=Path('./hypergraph'))

Begin to load multi omics data {'expr': './data/PEA_STA/expression_data.csv', 'protein': './data/PEA_STA/protein_data.csv'}
Load data done, time: 0.010265827178955078
Rename columns
Name expr Columns Index(['AKT1_expr', 'APC_expr', 'APP_expr', 'AURKB_expr', 'AXIN1_expr',
       'AXIN2_expr', 'BAX_expr', 'BCL2_expr', 'BCL2L1_expr', 'BCL6_expr',
       ...
       'THY1_expr', 'TNFRSF10B_expr', 'TNFRSF1A_expr', 'TP53_expr',
       'TRADD_expr', 'TUBB3_expr', 'TWSG1_expr', 'VEGFA_expr', 'VIM_expr',
       'XIAP_expr'],
      dtype='object', length=141)
Name protein Columns Index(['AKT1_protein', 'APC_protein', 'APP_protein', 'AURKB_protein',
       'AXIN1_protein', 'AXIN2_protein', 'BAX_protein', 'BCL2_protein',
       'BCL2L1_protein', 'BCL6_protein',
       ...
       'THY1_protein', 'TNFRSF10B_protein', 'TNFRSF1A_protein', 'TP53_protein',
       'TRADD_protein', 'TUBB3_protein', 'TWSG1_protein', 'VEGFA_protein',
       'VIM_protein', 'XIAP_protein'],
      dtype='object', length=141)
Un

In [9]:
process_data({
        'atac': './data/sci_CAR/ATAC_lsi.csv',
        'rna': './data/sci_CAR/RNA_pca.csv',
    }, cell_key='Unnamed: 0', dataname='sci_CAR', prefix=Path('./hypergraph'))

Begin to load multi omics data {'atac': './data/sci_CAR/ATAC_lsi.csv', 'rna': './data/sci_CAR/RNA_pca.csv'}
Load data done, time: 0.1323089599609375
Rename columns
Name atac Columns Index(['LSI_1_atac', 'LSI_2_atac', 'LSI_3_atac', 'LSI_4_atac', 'LSI_5_atac',
       'LSI_6_atac', 'LSI_7_atac', 'LSI_8_atac', 'LSI_9_atac', 'LSI_10_atac',
       'LSI_11_atac', 'LSI_12_atac', 'LSI_13_atac', 'LSI_14_atac',
       'LSI_15_atac', 'LSI_16_atac', 'LSI_17_atac', 'LSI_18_atac',
       'LSI_19_atac', 'LSI_20_atac', 'LSI_21_atac', 'LSI_22_atac',
       'LSI_23_atac', 'LSI_24_atac', 'LSI_25_atac', 'LSI_26_atac',
       'LSI_27_atac', 'LSI_28_atac', 'LSI_29_atac', 'LSI_30_atac',
       'LSI_31_atac', 'LSI_32_atac', 'LSI_33_atac', 'LSI_34_atac',
       'LSI_35_atac', 'LSI_36_atac', 'LSI_37_atac', 'LSI_38_atac',
       'LSI_39_atac', 'LSI_40_atac', 'LSI_41_atac', 'LSI_42_atac',
       'LSI_43_atac', 'LSI_44_atac', 'LSI_45_atac', 'LSI_46_atac',
       'LSI_47_atac', 'LSI_48_atac', 'LSI_49_atac', 'LSI_50_

In [10]:
process_data({
        'expr': './data/scNMT/expression_data_300.csv',
        'promoter_acc': './data/scNMT/promoter_acc_data_300.csv',
        'promoter_methy': './data/scNMT/promoter_methy_data_300.csv',
    }, cell_key='Unnamed: 0', dataname='scNMT', prefix=Path('./hypergraph'), read_csv_args={'header': None, 'sep': ' '})

Begin to load multi omics data {'expr': './data/scNMT/expression_data_300.csv', 'promoter_acc': './data/scNMT/promoter_acc_data_300.csv', 'promoter_methy': './data/scNMT/promoter_methy_data_300.csv'}
Load data done, time: 0.20604586601257324
Rename columns
Name expr Columns Index(['0_expr', '1_expr', '2_expr', '3_expr', '4_expr', '5_expr', '6_expr',
       '7_expr', '8_expr', '9_expr',
       ...
       '290_expr', '291_expr', '292_expr', '293_expr', '294_expr', '295_expr',
       '296_expr', '297_expr', '298_expr', '299_expr'],
      dtype='object', length=300)
Name promoter_acc Columns Index(['0_promoter_acc', '1_promoter_acc', '2_promoter_acc', '3_promoter_acc',
       '4_promoter_acc', '5_promoter_acc', '6_promoter_acc', '7_promoter_acc',
       '8_promoter_acc', '9_promoter_acc',
       ...
       '290_promoter_acc', '291_promoter_acc', '292_promoter_acc',
       '293_promoter_acc', '294_promoter_acc', '295_promoter_acc',
       '296_promoter_acc', '297_promoter_acc', '298_promote

In [11]:
process_data({
        'expr': './data/SCoPE2/expression_data.csv',
        'protein': './data/SCoPE2/protein_data.csv',
    }, cell_key='Unnamed: 0', dataname='SCoPE2', prefix=Path('./hypergraph'))

Begin to load multi omics data {'expr': './data/SCoPE2/expression_data.csv', 'protein': './data/SCoPE2/protein_data.csv'}
Load data done, time: 0.8141188621520996
Rename columns
Name expr Columns Index(['A0A075B6H9_expr', 'A0A0B4J1V0_expr', 'A0A0B4J237_expr',
       'A0A1B0GTH6_expr', 'A0A1B0GUA6_expr', 'A0A1B0GUU1_expr',
       'A0A1B0GUW6_expr', 'A0AVF1_expr', 'A0AVT1_expr', 'A0M8Q6_expr',
       ...
       'Q9Y6F8_expr', 'Q9Y6H5_expr', 'Q9Y6I0_expr', 'Q9Y6M7_expr',
       'Q9Y6N5_expr', 'Q9Y6R9_expr', 'Q9Y6U7_expr', 'Q9Y6W6_expr',
       'Q9Y6X6_expr', 'Q9Y6Z7_expr'],
      dtype='object', length=3042)
Name protein Columns Index(['A0A075B6H9_protein', 'A0A0B4J1V0_protein', 'A0A0B4J237_protein',
       'A0A1B0GTH6_protein', 'A0A1B0GUA6_protein', 'A0A1B0GUU1_protein',
       'A0A1B0GUW6_protein', 'A0AVF1_protein', 'A0AVT1_protein',
       'A0M8Q6_protein',
       ...
       'Q9Y6F8_protein', 'Q9Y6H5_protein', 'Q9Y6I0_protein', 'Q9Y6M7_protein',
       'Q9Y6N5_protein', 'Q9Y6R9_protein

In [12]:
process_data({
        'atac': './data/SNARE_seq_adult_mouse/ATAC_lsi.csv',
        'rna': './data/SNARE_seq_adult_mouse/RNA_pca.csv',
    }, cell_key='Unnamed: 0', dataname='SNARE_seq_adult_mouse', prefix=Path('./hypergraph'))

Begin to load multi omics data {'atac': './data/SNARE_seq_adult_mouse/ATAC_lsi.csv', 'rna': './data/SNARE_seq_adult_mouse/RNA_pca.csv'}
Load data done, time: 0.1284041404724121
Rename columns
Name atac Columns Index(['LSI_1_atac', 'LSI_2_atac', 'LSI_3_atac', 'LSI_4_atac', 'LSI_5_atac',
       'LSI_6_atac', 'LSI_7_atac', 'LSI_8_atac', 'LSI_9_atac', 'LSI_10_atac',
       'LSI_11_atac', 'LSI_12_atac', 'LSI_13_atac', 'LSI_14_atac',
       'LSI_15_atac', 'LSI_16_atac', 'LSI_17_atac', 'LSI_18_atac',
       'LSI_19_atac', 'LSI_20_atac', 'LSI_21_atac', 'LSI_22_atac',
       'LSI_23_atac', 'LSI_24_atac', 'LSI_25_atac', 'LSI_26_atac',
       'LSI_27_atac', 'LSI_28_atac', 'LSI_29_atac', 'LSI_30_atac',
       'LSI_31_atac', 'LSI_32_atac', 'LSI_33_atac', 'LSI_34_atac',
       'LSI_35_atac', 'LSI_36_atac', 'LSI_37_atac', 'LSI_38_atac',
       'LSI_39_atac', 'LSI_40_atac', 'LSI_41_atac', 'LSI_42_atac',
       'LSI_43_atac', 'LSI_44_atac', 'LSI_45_atac', 'LSI_46_atac',
       'LSI_47_atac', 'LSI_48_at

In [13]:
process_data({
        'atac': './data/SNARE_seq_neonatal_mouse/ATAC_lsi.csv',
        'rna': './data/SNARE_seq_neonatal_mouse/RNA_pca.csv',
    }, cell_key='Unnamed: 0', dataname='SNARE_seq_neonatal_mouse', prefix=Path('./hypergraph'))

Begin to load multi omics data {'atac': './data/SNARE_seq_neonatal_mouse/ATAC_lsi.csv', 'rna': './data/SNARE_seq_neonatal_mouse/RNA_pca.csv'}
Load data done, time: 0.06936812400817871
Rename columns
Name atac Columns Index(['LSI_1_atac', 'LSI_2_atac', 'LSI_3_atac', 'LSI_4_atac', 'LSI_5_atac',
       'LSI_6_atac', 'LSI_7_atac', 'LSI_8_atac', 'LSI_9_atac', 'LSI_10_atac',
       'LSI_11_atac', 'LSI_12_atac', 'LSI_13_atac', 'LSI_14_atac',
       'LSI_15_atac', 'LSI_16_atac', 'LSI_17_atac', 'LSI_18_atac',
       'LSI_19_atac', 'LSI_20_atac', 'LSI_21_atac', 'LSI_22_atac',
       'LSI_23_atac', 'LSI_24_atac', 'LSI_25_atac', 'LSI_26_atac',
       'LSI_27_atac', 'LSI_28_atac', 'LSI_29_atac', 'LSI_30_atac',
       'LSI_31_atac', 'LSI_32_atac', 'LSI_33_atac', 'LSI_34_atac',
       'LSI_35_atac', 'LSI_36_atac', 'LSI_37_atac', 'LSI_38_atac',
       'LSI_39_atac', 'LSI_40_atac', 'LSI_41_atac', 'LSI_42_atac',
       'LSI_43_atac', 'LSI_44_atac', 'LSI_45_atac', 'LSI_46_atac',
       'LSI_47_atac', 'LS

In [14]:
process_data({
        'expr': './data/CITE_seq/expression_data.csv',
        'protein': './data/CITE_seq/protein_data.csv',
    }, cell_key='cell ID', dataname='CITE_seq', prefix=Path('./hypergraph'))

Begin to load multi omics data {'expr': './data/CITE_seq/expression_data.csv', 'protein': './data/CITE_seq/protein_data.csv'}


KeyboardInterrupt: 